##### Runnable PySpark example showing how to handle corrupt or malformed records during ingestion, using the built-in _corrupt_record column.

- We’ll use PERMISSIVE mode so Spark loads valid rows while capturing bad ones for later inspection.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType,IntegerType,FloatType,StructType,StructField

In [0]:
spark = SparkSession.builder.appName("Handling Bad Records").getOrCreate()

In [0]:
# -----------------------------
# 2. Sample JSON Data (with corrupt rows)
# -----------------------------
data = [
    '{"name": "Alice", "age": 30}',       # valid
    '{"name": "Bob", "age": "thirty"}',   # invalid: age should be int
    '{"name": "Charlie"}',                # missing age (still valid if nullable)
    '{"name": "David", "age": 25}',       # valid
    '{"name": "Eve", "age": 40',          # corrupt JSON (missing closing brace)
]


In [0]:
# Save sample data to a local file
input_path = "/Workspace/Users/pdkusalkar@gmail.com/databricks_work/handling_json_data/sample_data.json"
with open(input_path, "w") as f:
    for line in data:
        f.write(line + "\n")

In [0]:
schema = StructType([
    StructField("name",StringType(),True),
    StructField("age",StringType(),True),
    StructField("_corrupt_record",StringType(),True)
])

In [0]:
# -----------------------------
# 4. Read JSON with PERMISSIVE mode
# -----------------------------
df = spark.read.format("json") \
               .option("mode","PERMISSIVE") \
               .option("schema",schema) \
               .option("columnNameOfCorruptRecord","_corrupt_record") \
               .load(input_path)

display(df)

In [0]:
# -----------------------------
# 5. Separate valid and corrupt records
# -----------------------------
valid_df = df.filter(df["_corrupt_record"].isNull())
corrupt_df = df.filter(df["_corrupt_record"].isNotNull())

# -----------------------------
# 6. Show results
# -----------------------------
print("✅ Valid Records:")
valid_df.show(truncate=False)

print("⚠️ Corrupt Records:")
corrupt_df.show(truncate=False)